# ⚙️ Chat State Architecture: Token Inflation, Memory Windows, and Summarization Compression

**My Objective:** I am auditing conversational state management for LLM pipelines. Because LLM APIs are inherently stateless, passing raw conversation history natively scales token costs linearly and inevitably exhausts the context window. I evaluate two enterprise memory architectures—Sliding Windows and Summarization Compression—to balance token expenditure against factual recall.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "prompting").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "prompting"))
sys.path.insert(0, str(ROOT / "orchestration"))

from dotenv import load_dotenv
load_dotenv(override=True)

import os
from reasoning_utils import DiskCache, Lab
import openai

client = openai.OpenAI(api_key=os.environ["API_KEY"], base_url=os.environ["BASE_URL"])
MODEL = os.environ["MODEL"]
cache = DiskCache(Path.home() / "Desktop" / "FSDS" / "gen_ai" / "applied-llm-ops" / "chat_memory_cache.json")
lab = Lab(client, MODEL, cache=cache)

from chat_tasks import BANKASSIST_SYSTEM, RECALL_PROBE, RECALL_TOKEN, SUMMARY_EVERY, WINDOW_SIZE
from chat_utils import (append_turn, chat_memory_markdown, check_chat_memory_work, compare_strategies, parse_chat_report,
                        recall_succeeded, run_strategy, save_chat_report, slide_window, strategy_comparison_table,
                        summarize_older, turn_cost_table)

print(f"model: {MODEL}")
print(f"conversation length: {len(RECALL_PROBE)} turns")
print(f"planted fact (recall token): {RECALL_TOKEN!r}, planted at turn 2, asked for again at turn {len(RECALL_PROBE)}")

## 1. The Statelessness Constraint (Baseline Failure)
LLM endpoints have zero native memory. To prove this, I isolated turn 17 of a scripted conversation (requesting a reference number established in turn 2) and sent it as a raw, single-turn inference call to verify failure.

In [ ]:
stateless_messages = [
    {"role": "system", "content": BANKASSIST_SYSTEM},
    {"role": "user", "content": RECALL_PROBE[-1]},   
]
r = lab.chat(stateless_messages)
print(r["text"])
print()
print("recall token present:", recall_succeeded(r["text"]))

**Architectural Insight — Explicit Injection:** The model failed because it had no context. What users perceive as a chatbot "remembering" is actually the orchestration code explicitly injecting a growing transcript of prior interactions into the API payload on every single request.

In [ ]:
messages = [{"role": "system", "content": BANKASSIST_SYSTEM}]

messages = append_turn(messages, "user", RECALL_PROBE[0])
r1 = lab.chat(messages)
messages = append_turn(messages, "assistant", r1["text"].strip())

messages = append_turn(messages, "user", RECALL_PROBE[1])   
r2 = lab.chat(messages)
messages = append_turn(messages, "assistant", r2["text"].strip())

messages = append_turn(messages, "user", RECALL_PROBE[-1])  
r3 = lab.chat(messages)
print(r3["text"])
print()
print("recall token present:", recall_succeeded(r3["text"]))
print("messages in the call:", len(messages) + 1)

## 2. Naive State Management & Token Inflation
I executed the entire 17-turn baseline conversation using a naive memory strategy, appending every turn to the payload continuously.

In [ ]:
naive_result = run_strategy(lab, "naive")

print(naive_result["final_reply"])
print()
print("recall succeeded:", naive_result["recall_ok"])
print("turns played:", naive_result["n_turns"])

**Telemetry Benchmark — The Inflation Curve:** While naive state management perfectly preserves factual recall (100%), it incurs compounding compute costs because turn 17 pays for the tokens generated in turns 1 through 16, regardless of relevance. I visualized this cost curve below.

In [ ]:
print(turn_cost_table(naive_result))
print()
first, last = naive_result["turn_costs"][0], naive_result["turn_costs"][-1]
print(f"turn 1 cost: {first:,} tokens")
print(f"turn {naive_result['n_turns']} cost: {last:,} tokens")
print(f"growth: {last / first:.1f}x, for a conversation that only grew {naive_result['n_turns']}x in NUMBER of turns")

In [ ]:
costs = naive_result["turn_costs"]
n = len(costs)
avg_growth_per_turn = (costs[-1] - costs[0]) / (n - 1)
projected_turn_100 = costs[-1] + avg_growth_per_turn * (100 - n)

print(f"measured growth: about {avg_growth_per_turn:,.0f} tokens per additional turn")
print(f"projected cost at turn 100, same rate: about {projected_turn_100:,.0f} tokens for ONE call")
print()
print("Two separate walls, in order: (1) that single call gets expensive long before any ceiling is reached —")
print("real, compounding spend, every call, on a long-running assistant. (2) every model ALSO has a hard token")
print("ceiling (check your provider's model page for this model's exact number) — past it, a call does not")
print("degrade, it fails outright, with no partial answer.")

## 3. Architecture Fix 1: Sliding Window Discard
To flatten the token inflation curve, I implemented a rolling cutoff that automatically truncates messages falling outside the active `WINDOW_SIZE` parameter.

In [ ]:
window_result = run_strategy(lab, "window", window_size=WINDOW_SIZE)

print(window_result["final_reply"])
print()
print("recall succeeded:", window_result["recall_ok"])
print()
print(turn_cost_table(window_result))

**Optimization Trade-off — Silent Forgetting:** The sliding window successfully caps inference costs, maintaining a flat trajectory after the window fills. However, it completely sacrifices long-term factual recall. Any variable generated outside the fixed window is permanently lost without warning to the application or the user.

## 4. Architecture Fix 2: Compression & Summarization
To recover the factual recall lost by truncation, I implemented periodic compression via a secondary LLM call. This takes older turns and condenses them into a system-injected summary string.

In [ ]:
summarize_result = run_strategy(lab, "summarize", window_size=WINDOW_SIZE, summary_every=SUMMARY_EVERY)

print(summarize_result["final_reply"])
print()
print("recall succeeded:", summarize_result["recall_ok"])
print("extra model calls spent summarizing:", summarize_result["extra_calls"])
print()
print(turn_cost_table(summarize_result))

## 5. Architectural A/B/C Benchmarking
I ran all three architectures side-by-side to construct a deterministic deployment matrix comparing API cost overheads against factual retention.

In [ ]:
results = compare_strategies(lab, window_size=WINDOW_SIZE, summary_every=SUMMARY_EVERY)
print(strategy_comparison_table(results))

## 6. Serialization & CI/CD Checkpoint
No architecture optimizes both token costs and recall perfectly; selecting the appropriate state manager requires analyzing the specific business logic. I serialized this benchmark to my CI/CD pipeline repository to track historical token degradation.

In [ ]:
WORK_REPO = r"C:\Users\HP\Desktop\FSDS\gen_ai\applied-llm-ops"   # <-- change this to YOUR folder

report_text = chat_memory_markdown(results, MODEL)
print(report_text)

In [ ]:
from prompt_utils import results_table

checkpoint_results = check_chat_memory_work(WORK_REPO)
print(results_table(checkpoint_results))